In [1]:
import sys

# 确保可以找到项目根目录下的依赖（dnn、mydatasets 等）
sys.path.append("..")

from SpatialAlign.train_sc_stage1 import train_model
from SpatialAlign.pseudo_labeling_impl import pseudoing_label
from SpatialAlign.train_stage2 import train_for_stage2

import scanpy as sc
import numpy as np
import pandas as pd


In [2]:
# 路径配置（基于原 nsclc.ipynb）
adata_sc_path = "/public/home/singlecell/users/liuchunlong/scai/stAI/mymodel/Datasets/NLCSC/original/0729/sc_nsclc.h5ad"
adata_st_path = "/public/home/singlecell/users/liuchunlong/scai/stAI/mymodel/Datasets/NLCSC/original/0729/lung9_rep1_15types.h5ad"

# Stage 1 SC 分类器 checkpoint
stage1_ckpt_path = "/public/home/singlecell/users/liuchunlong/scai/stAI/mymodel/checkpoint/NSCLC/0731/SC/mlp_stage1_1114.pt"

# Stage 2 GAT encoder checkpoint
gat_pt_path = "/public/home/singlecell/users/liuchunlong/scai/stAI/mymodel/checkpoint/NSCLC/0731/ST/gat_st.pt"

# Stage 2 中更新后的 MLP checkpoint（与 stage1 复用）
mlp_stage2_save_path = stage1_ckpt_path

# 预先计算好的 UMAP 坐标（与原 nsclc.ipynb 相同文件）
umap_csv_path = "/public/home/singlecell/users/liuchunlong/scai/stAI/mymodel/0729/X_umap_df.csv"


In [3]:
# Stage 1：训练 scRNA‑seq 分类器，并返回对齐后的 adata_sc / adata_st
adata_sc, adata_st, prototypes = train_model(
    adata_sc_path,
    adata_st_path,
    stage1_ckpt_path,
)
adata_st

Epoch 1/7 | Loss: 122.9504 (cls 122.9504  | Train Acc: 0.3356 | Val Acc: 0.1501
Epoch 2/7 | Loss: 92.0118 (cls 92.0118  | Train Acc: 0.5432 | Val Acc: 0.6568
Epoch 3/7 | Loss: 60.5125 (cls 60.5125  | Train Acc: 0.7375 | Val Acc: 0.6857
Epoch 4/7 | Loss: 38.8334 (cls 38.8334  | Train Acc: 0.8272 | Val Acc: 0.7047
Epoch 5/7 | Loss: 25.8135 (cls 25.8135  | Train Acc: 0.8765 | Val Acc: 0.7073
Epoch 6/7 | Loss: 19.6163 (cls 19.6163  | Train Acc: 0.8958 | Val Acc: 0.7083
Epoch 7/7 | Loss: 16.0975 (cls 16.0975  | Train Acc: 0.9099 | Val Acc: 0.7051
Label: B-cell, Prototype: [ 4.3261936e-01  1.3397108e-02 -7.5258613e-02  1.3393617e+00
  3.3624420e+00  2.3730120e-01  1.1896561e+00  1.3722961e-01
  2.7072376e-01 -4.1248277e-02  6.9558388e-01  1.4236203e+00
 -4.2026907e-02  1.7782328e+00 -6.9390632e-02  6.6833001e-01
  2.7826617e+00 -1.1089464e-01  1.2087955e+00  3.6455188e+00
 -1.2159431e-01  5.5568647e+00  4.0676470e+00 -9.2595562e-02
  3.8731592e+00  2.1737721e+00  2.8892386e-01  3.2770579e+00

AnnData object with n_obs × n_vars = 85904 × 952
    obs: 'fov', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'celltype'
    uns: 'log1p'
    obsm: 'spatial'

In [4]:
# 使用 Stage 1 模型对 ST 做伪标签推断
adata_st = pseudoing_label(adata_st, stage1_ckpt_path)
adata_st

Filtered Accuracy: 0.7083
              precision    recall  f1-score   support

       T CD4     0.3396    0.0580    0.0990      4036
         mDC     0.0000    0.0000    0.0000       994
         pDC     0.2857    0.0075    0.0146      1065
      B-cell     0.3456    0.3892    0.3661       871
  epithelial     0.3440    0.4088    0.3736      3584
        mast     0.7653    0.2577    0.3856       291
  neutrophil     0.7306    0.0237    0.0459      5948
  fibroblast     0.6168    0.9794    0.7569     13952
      tumors     0.8231    0.9474    0.8809     39422
 endothelial     0.6452    0.5799    0.6108      4428
          NK     0.4800    0.0483    0.0877       746
  macrophage     0.6221    0.7522    0.6810      4080
 plasmablast     0.7010    0.5036    0.5861      2895
        Treg     0.3421    0.1545    0.2128      1269
       T CD8     0.5021    0.1042    0.1725      2323

    accuracy                         0.7083     85904
   macro avg     0.5029    0.3476    0.3516     85904


AnnData object with n_obs × n_vars = 85904 × 952
    obs: 'fov', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'celltype', 'pseudo_label', 'pseudo_confidence'
    uns: 'log1p'
    obsm: 'spatial'

In [5]:
# 附加预先计算好的 UMAP 坐标（与原 nsclc.ipynb 逻辑一致）
umap_df = pd.read_csv(umap_csv_path, index_col=0)
shared = adata_st.obs_names.intersection(umap_df.index)
adata_st = adata_st[shared].copy()
adata_st.obsm["X_umap"] = umap_df.loc[shared, ["UMAP1", "UMAP2"]].to_numpy()
adata_st

AnnData object with n_obs × n_vars = 85904 × 952
    obs: 'fov', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'celltype', 'pseudo_label', 'pseudo_confidence'
    uns: 'log1p'
    obsm: 'spatial', 'X_umap'

In [6]:
# Stage 2：基于伪标签训练空间 encoder（GAT）
adata_st = train_for_stage2(
    stage1_ckpt_path,
    adata_sc,
    adata_st,
    gat_pt_path,
    mlp_stage2_save_path,
)
adata_st


===== Epoch 1 pseudo_confidence stats by class =====
Class          B-cell | n=   981 | mean=0.513, std=0.234, p25=0.318, p50=0.470, p75=0.699, min=0.116, max=0.991
Class              NK | n=    75 | mean=0.451, std=0.206, p25=0.317, p50=0.425, p75=0.570, min=0.113, max=0.960
Class           T CD4 | n=   689 | mean=0.499, std=0.203, p25=0.332, p50=0.484, p75=0.645, min=0.106, max=0.963
Class           T CD8 | n=   482 | mean=0.563, std=0.236, p25=0.368, p50=0.533, p75=0.767, min=0.125, max=0.997
Class            Treg | n=   573 | mean=0.439, std=0.193, p25=0.295, p50=0.409, p75=0.546, min=0.101, max=0.986
Class     endothelial | n=  3980 | mean=0.642, std=0.272, p25=0.398, p50=0.657, p75=0.918, min=0.101, max=1.000
Class      epithelial | n=  4259 | mean=0.577, std=0.212, p25=0.413, p50=0.565, p75=0.741, min=0.105, max=0.998
Class      fibroblast | n= 22154 | mean=0.839, std=0.219, p25=0.725, p50=0.965, p75=0.998, min=0.104, max=1.000
Class             mDC | n=     5 | mean=0.206, std

AnnData object with n_obs × n_vars = 85904 × 952
    obs: 'fov', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'celltype', 'pseudo_label', 'pseudo_confidence'
    uns: 'log1p'
    obsm: 'spatial', 'X_umap'